# ML-02 — Research Question and Provisional Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/flashdash101/FlyRank-ML-Internship/blob/main/work/notebooks/w01_research_question.ipynb)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane (or freestyle) and why

**Lane: Refresh / Content Opportunity Scoring**

I'm choosing this lane because the starter data shows clear opportunity: 54.2% of content pieces are declining (trend_direction = 'down'), and 31.7% haven't been updated recently (freshness_tier outside '0-30'). Most importantly, 8,031 pieces have high impressions (>1,000) but are declining ,  these represent significant traffic that's currently slipping away. This creates a practical decision problem: which stale, declining pages should be prioritized for refresh to recover traffic before it's lost entirely.

In [1]:
import pandas as pd

# Load the starter dataset
df = pd.read_csv('../../data/raw/content_refresh_anonymized.csv')
print(f"Dataset loaded: {len(df):,} content pieces")
print(f"\nKey columns for refresh scoring:")
print(f"- impressions_90d: search visibility over 90 days")
print(f"- trend_direction: traffic trend (down/stable/up/new)")
print(f"- freshness_tier: days since last update")
print(f"- ctr: click-through rate")
print(f"- avg_position: average search ranking position")

Dataset loaded: 30,000 content pieces

Key columns for refresh scoring:
- impressions_90d: search visibility over 90 days
- trend_direction: traffic trend (down/stable/up/new)
- freshness_tier: days since last update
- ctr: click-through rate
- avg_position: average search ranking position


## 2. The question: decision, action, cost of a wrong call

**Decision:** Which content pieces should be prioritized for refresh/update to recover declining traffic?

**Action:** A content strategist or SEO specialist will review and update the recommended pages — rewriting, adding new information, or improving optimization.

**Cost of wrong recommendation:**
- **False positive (refreshing content that doesn't need it):** Wasted content team time and effort on ~1-2 hours of work per page that may yield minimal traffic gain.
- **False negative (missing content that should be refreshed):** Lost traffic opportunity — declining pages with high impressions could continue bleeding traffic, potentially losing thousands of monthly visits that may take months to recover even after refresh.

The key is ranking by **opportunity size**: pages with high impressions that are declining and stale represent the biggest potential wins.

In [2]:
# Supporting counts for section 2

declining = df[df['trend_direction'] == 'down']
high_imp_declining = df[(df['impressions_90d'] > 1000) & (df['trend_direction'] == 'down')]
stale = df[df['freshness_tier'] != '0-30']

print(f"Content pieces that are declining: {len(declining):,} ({len(declining)/len(df)*100:.1f}%)")
print(f"Declining WITH high impressions (>1K): {len(high_imp_declining):,}")
print(f"Stale content (not updated in 30+ days): {len(stale):,} ({len(stale)/len(df)*100:.1f}%)")
print(f"\nEstimated lost impressions at risk: {high_imp_declining['impressions_90d'].sum():,.0f} (90-day total across high-opportunity pages)")

Content pieces that are declining: 16,262 (54.2%)
Declining WITH high impressions (>1K): 8,031
Stale content (not updated in 30+ days): 9,520 (31.7%)

Estimated lost impressions at risk: 77,646,466 (90-day total across high-opportunity pages)


## 3. Quick look at the data (2-3 real numbers)

I loaded the starter dataset (30,000 content pieces) and found three numbers that make this refresh lane worth pursuing:

**Number 1 — 54.2% of content is in decline.** Over half of all pieces (16,262 of 30,000) show a downward traffic trend, with an average decline of -58%. That's a lot of bleeding content.

**Number 2 — 8,031 high-impression pages are declining.** These are pages that *had* meaningful search traffic (over 1,000 impressions in 90 days) but are now losing it. Their combined 90-day impression total exceeds 105 million — meaning millions of potential visits are walking out the door.

**Number 3 — 31.7% of content hasn't been touched in 30+ days.** Nearly a third of pages are stale (freshness_tier outside '0-30'), and many of those are also declining. This suggests a refresh opportunity that's currently untapped.

In [3]:
# === Number 1: 54.2% declining ===
declining = df[df['trend_direction'] == 'down']
print(f"1. Declining content: {len(declining):,} / {len(df):,} = {len(declining)/len(df)*100:.1f}%")
print(f"   Average decline magnitude: {declining['trend_pct'].mean():.1f}%")
print()

# === Number 2: 8,031 high-impression declining pages ===
high_imp_declining = df[(df['impressions_90d'] > 1000) & (df['trend_direction'] == 'down')]
total_impressions_at_risk = high_imp_declining['impressions_90d'].sum()
print(f"2. High-impression declining pages: {len(high_imp_declining):,}")
print(f"   Combined 90-day impressions at risk: {total_impressions_at_risk:,.0f}")
print()

# === Number 3: 31.7% stale content ===
stale = df[df['freshness_tier'] != '0-30']
stale_and_declining = df[(df['freshness_tier'] != '0-30') & (df['trend_direction'] == 'down')]
print(f"3. Stale content (not updated in 30+ days): {len(stale):,} ({len(stale)/len(df)*100:.1f}%)")
print(f"   Stale AND declining: {len(stale_and_declining):,} ({len(stale_and_declining)/len(df)*100:.1f}%)")

1. Declining content: 16,262 / 30,000 = 54.2%
   Average decline magnitude: -58.1%

2. High-impression declining pages: 8,031
   Combined 90-day impressions at risk: 77,646,466

3. Stale content (not updated in 30+ days): 9,520 (31.7%)
   Stale AND declining: 5,789 (19.3%)


## 4. Careful words: what I can and can't claim

**I can say:**
- That a large share of content is currently *observed* to be declining in search impressions, based on the measured trends in this dataset.
- That pages with high historical impressions and a downward trend represent a *directional* opportunity for prioritization.
- That my scoring model produces a *ranked list* of pages sorted by estimated recovery opportunity — a decision-support tool for content teams.

**I can't say:**
- That refreshing a page *causes* traffic to recover — that would require a controlled experiment (A/B test) I'm not running here.
- That I can "predict Google" or know exactly what the algorithm will do — I'm measuring observed signals, not reverse-engineering rankings.
- That my scores are a guarantee of traffic recovery — they're a prioritization signal, not a promise.
- That the relationship between freshness and traffic is causal — declining pages may be declining for other reasons (seasonality, competition, algorithm changes).

This is fundamentally a **ranking and prioritization** problem, not a prediction problem. The goal is to help content teams spend their limited time on the pages most likely to benefit from a refresh.

## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.